## CYK and P-CYK parser

In [71]:
from collections import defaultdict
from nltk import Tree
import nltk

# Ensure nltk resources are available
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
# ---------------------------
# CYK PARSER (Non-Probabilistic)
# ---------------------------
def cyk_parser(grammar, tokens):
    n = len(tokens)
    table = [[set() for _ in range(n)] for _ in range(n)]
    back = [[defaultdict(list) for _ in range(n)] for _ in range(n)]

    # Inverse grammar mapping: RHS -> LHS
    rhs_to_lhs = defaultdict(set)
    for lhs, rules in grammar.items():
        for rhs in rules:
            rhs_to_lhs[tuple(rhs)].add(lhs)

    # Fill diagonals
    for i, token in enumerate(tokens):
        for lhs in rhs_to_lhs.get((token,), []):
            table[i][i].add(lhs)

    # Fill upper triangle
    for l in range(2, n+1):
        for i in range(n - l + 1):
            j = i + l - 1
            for k in range(i, j):
                for B in table[i][k]:
                    for C in table[k+1][j]:
                        for A in rhs_to_lhs.get((B, C), []):
                            table[i][j].add(A)
                            back[i][j][A].append((k, B, C))

    # Recursive tree building
    def build_tree(i, j, symbol):
        if i == j:
            return (symbol, tokens[i])
        for k, B, C in back[i][j].get(symbol, []):
            left = build_tree(i, k, B)
            right = build_tree(k+1, j, C)
            return (symbol, left, right)

    if 'S' in table[0][n-1]:
        return build_tree(0, n-1, 'S')
    else:
        return None

In [ ]:
# ---------------------------
# EXAMPLE USAGE
# ---------------------------
tokens = ['the', 'cat', 'chased', 'the', 'mouse']

# CYK grammar
grammar = {
    'S': [['NP', 'VP']],
    'VP': [['V', 'NP']],
    'NP': [['Det', 'N']],
    'Det': [['the']],
    'N': [['cat'], ['mouse']],
    'V': [['chased']]
}

# Run CYK
cyk_tree = cyk_parser(grammar, tokens)
if cyk_tree:
    print("CYK Parse Tree (tuple):")
    print(cyk_tree)
    nltk_tree = tuple_to_nltk_tree(cyk_tree)
    nltk_tree.pretty_print()
else:
    print("No parse tree found.Invalid input sentence")

CYK Parse Tree (tuple):
('S', ('NP', ('Det', 'the'), ('N', 'cat')), ('VP', ('V', 'chased'), ('NP', ('Det', 'the'), ('N', 'mouse'))))
              S                 
      ________|_____             
     |              VP          
     |         _____|___         
     NP       |         NP      
  ___|___     |      ___|____    
Det      N    V    Det       N  
 |       |    |     |        |   
the     cat chased the     mouse



In [ ]:
# ---------------------------
# PROBABILISTIC CYK PARSER
# ---------------------------
def pcyk_parser(grammar, tokens):
    n = len(tokens)
    table = [[defaultdict(float) for _ in range(n)] for _ in range(n)]
    back = [[{} for _ in range(n)] for _ in range(n)]

    for i, token in enumerate(tokens):
        for lhs in grammar:
            for rhs, prob in grammar[lhs]:
                if len(rhs) == 1 and rhs[0] == token:
                    if prob > table[i][i][lhs]:
                        table[i][i][lhs] = prob
                        back[i][i][lhs] = token

    for l in range(2, n+1):
        for i in range(n - l + 1):
            j = i + l - 1
            for k in range(i, j):
                for A in grammar:
                    for (rhs, prob) in grammar[A]:
                        if len(rhs) == 2:
                            B, C = rhs
                            prob_B = table[i][k].get(B, 0)
                            prob_C = table[k+1][j].get(C, 0)
                            if prob_B and prob_C:
                                new_prob = prob * prob_B * prob_C
                                if new_prob > table[i][j].get(A, 0):
                                    table[i][j][A] = new_prob
                                    back[i][j][A] = (k, B, C)

    def build_tree(i, j, symbol):
        if i == j:
            return (symbol, back[i][j][symbol])
        k, B, C = back[i][j][symbol]
        return (symbol, build_tree(i, k, B), build_tree(k+1, j, C))

    if 'S' in table[0][n-1]:
        return build_tree(0, n-1, 'S')
    else:
        return None

In [ ]:
# ---------------------------
# CONVERT TO nltk.Tree & DISPLAY
# ---------------------------
def tuple_to_nltk_tree(tree_tuple):
    if isinstance(tree_tuple, tuple):
        label = tree_tuple[0]
        children = [tuple_to_nltk_tree(child) for child in tree_tuple[1:]]
        return Tree(label, children)
    else:
        return tree_tuple

In [ ]:
# PCYK grammar
p_grammar = {
    'S': [ (['NP', 'VP'], 1.0) ],
    'VP': [ (['V', 'NP'], 1.0) ],
    'NP': [ (['Det', 'N'], 1.0) ],
    'Det': [ (['the'], 1.0) ],
    'N': [ (['cat'], 0.5), (['mouse'], 0.5) ],
    'V': [ (['chased'], 1.0) ]
}

# Run PCYK
pcyk_tree = pcyk_parser(p_grammar, tokens)
if pcyk_tree:
    print("\nPCYK Parse Tree (tuple)- returns the most probable parse tree:")
    print(pcyk_tree)
    nltk_tree = tuple_to_nltk_tree(pcyk_tree)
    nltk_tree.pretty_print()
else:
    print("No parse tree found.Invalid input sentence")


PCYK Parse Tree (tuple)- returns the most probable parse tree:
('S', ('NP', ('Det', 'the'), ('N', 'cat')), ('VP', ('V', 'chased'), ('NP', ('Det', 'the'), ('N', 'mouse'))))
              S                 
      ________|_____             
     |              VP          
     |         _____|___         
     NP       |         NP      
  ___|___     |      ___|____    
Det      N    V    Det       N  
 |       |    |     |        |   
the     cat chased the     mouse



In [46]:
from collections import defaultdict


def extract_rules_from_tree(tree):
    rules = []
    if len(tree) == 2 and isinstance(tree[1], str):  # Terminal rule
        lhs, word = tree
        rules.append((lhs, (word,)))
    elif len(tree) == 3:
        lhs, left, right = tree
        rules.append((lhs, (left[0], right[0])))
        rules += extract_rules_from_tree(left)
        rules += extract_rules_from_tree(right)
    return rules

def compute_rule_probabilities(trees):
    rule_counts = defaultdict(int)
    lhs_counts = defaultdict(int)

    for tree in trees:
        rules = extract_rules_from_tree(tree)
        for lhs, rhs in rules:
            rule_counts[(lhs, rhs)] += 1
            lhs_counts[lhs] += 1

    rule_probs = {
        (lhs, rhs): count / lhs_counts[lhs]
        for (lhs, rhs), count in rule_counts.items()
    }

    return rule_probs

def collect_leaves(tree):
    if len(tree) == 2 and isinstance(tree[1], str):
        return [tree[1]]
    elif len(tree) == 3:
        return collect_leaves(tree[1]) + collect_leaves(tree[2])
    return []

def validate_and_score_tree(tree, sentence, rule_probs):
    leaves = collect_leaves(tree)
    matches = (leaves == sentence)

    return matches

def compute_tree_probability(tree, rule_probs):
    if len(tree) == 2 and isinstance(tree[1], str):
        lhs, word = tree
        rule = (lhs, (word,))
        if rule not in rule_probs:
            raise ValueError(f"Unknown rule: {lhs} -> '{word}'")
        return rule_probs[rule]
    elif len(tree) == 3:
        lhs, left, right = tree
        rule = (lhs, (left[0], right[0]))
        if rule not in rule_probs:
            raise ValueError(f"Unknown rule: {lhs} -> {left[0]} {right[0]}")
        left_prob = compute_tree_probability(left, rule_probs)
        right_prob = compute_tree_probability(right, rule_probs)
        return rule_probs[rule] * left_prob * right_prob
    else:
        raise ValueError("Invalid tree format")

training_trees = [
    ('S', ('NP', 'she'), ('VP', ('V', 'eats'), ('NP', 'fish'))),
    ('S', ('NP', 'fish'), ('VP', 'eats')),
    ('S', ('NP', 'she'), ('VP', 'eats')),
    ('S', ('NP', 'fish'), ('VP', ('V', 'eats'), ('NP', 'fish')))
]

rule_probs = compute_rule_probabilities(training_trees)

sentence = ['she', 'eats', 'fish']
test_tree = (
    'S',
    ('NP', 'she'),
    ('VP',
        ('V', 'eats'),
        ('NP', 'fish')
    )
)

is_valid = validate_and_score_tree(test_tree, sentence, rule_probs)

print("Sentence Valid for Tree?", is_valid)
print("\nLearned Grammar Rules with Probabilities:")
for (lhs, rhs), p in rule_probs.items():
    print(f"{lhs} -> {' '.join(rhs)} [{p:.3f}]")

Sentence Valid for Tree? True

Learned Grammar Rules with Probabilities:
S -> NP VP [1.000]
NP -> she [0.333]
VP -> V NP [0.500]
V -> eats [1.000]
NP -> fish [0.667]
VP -> eats [0.500]


## Machine Translation

Sep2Seq model (LSTM-LSTM)

In [59]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Dummy dataset
pairs = [
    ("i am a student", "je suis un étudiant"),
    ("he is a teacher", "il est un professeur"),
    ("she is happy", "elle est heureuse"),
    ("they are playing", "ils jouent"),
    ("you are smart", "tu es intelligent")
]

# Tokenize and build vocab
def tokenize(sentence):
    return sentence.lower().split()

def build_vocab(sentences):
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2}
    idx = 3
    for sent in sentences:
        for word in tokenize(sent):
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

src_vocab = build_vocab([src for src, tgt in pairs])
tgt_vocab = build_vocab([tgt for src, tgt in pairs])
inv_tgt_vocab = {v: k for k, v in tgt_vocab.items()}

def encode(sentence, vocab):
    return [vocab["<sos>"]] + [vocab[word] for word in tokenize(sentence)] + [vocab["<eos>"]]

data = [(encode(src, src_vocab), encode(tgt, tgt_vocab)) for src, tgt in pairs]

SRC_VOCAB_SIZE = len(src_vocab)
TGT_VOCAB_SIZE = len(tgt_vocab)
EMBED_SIZE = 32
HIDDEN_SIZE = 64


In [60]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim)    # IF U R USING RNN ---> self.rnn = nn.RNN(emb_dim, hid_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

        '''# RNN version
        outputs, hidden = self.rnn(embedded)
        return hidden, None'''

class Decoder(nn.Module):
    def __init__(self, output_dim, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim)    # IF U R USING RNN ----> self.rnn = nn.RNN(emb_dim, hid_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(0))
        return prediction, hidden, cell

        ''' # RNN version
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, None'''

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        tgt_len = tgt.shape[0]
        batch_size = 1
        tgt_vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(tgt_len, tgt_vocab_size)
        hidden, cell = self.encoder(src)

        input = tgt[0]  # <sos>
        for t in range(1, tgt_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1 = output.argmax(1)
            input = tgt[t] if random.random() < teacher_forcing_ratio else top1
        return outputs

In [61]:
encoder = Encoder(SRC_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
decoder = Decoder(TGT_VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE)
model = Seq2Seq(encoder, decoder)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Training loop
for epoch in range(100):
    total_loss = 0
    for src, tgt in data:
        src_tensor = torch.tensor(src).unsqueeze(1)  # (seq_len, 1)
        tgt_tensor = torch.tensor(tgt).unsqueeze(1)

        optimizer.zero_grad()
        output = model(src_tensor, tgt_tensor)

        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        tgt_tensor = tgt_tensor[1:].view(-1)

        loss = criterion(output, tgt_tensor)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 14.3868
Epoch 10, Loss: 11.0641
Epoch 20, Loss: 4.6747
Epoch 30, Loss: 2.0453
Epoch 40, Loss: 1.0558
Epoch 50, Loss: 0.6260
Epoch 60, Loss: 0.4141
Epoch 70, Loss: 0.2963
Epoch 80, Loss: 0.2242
Epoch 90, Loss: 0.1766


In [66]:
# --------------------------
# Inference
# --------------------------
def translate(model, sentence, max_len=10):
    model.eval()
    tokens = encode(sentence, src_vocab)
    src_tensor = torch.tensor(tokens).unsqueeze(1)

    hidden, cell = model.encoder(src_tensor)
    input = torch.tensor([tgt_vocab["<sos>"]])

    result = []
    for _ in range(max_len):
        output, hidden, cell = model.decoder(input, hidden, cell)
        top1 = output.argmax(1).item()
        if top1 == tgt_vocab["<eos>"]:
            break
        result.append(inv_tgt_vocab[top1])
        input = torch.tensor([top1])

    return ' '.join(result)


# Test translation
print("\nTranslation Examples:")
print("Input: 'i am happy'")
print("Output:", translate(model, "i am happy"))

print("Input: 'you are smart'")
print("Output:", translate(model, "you are smart"))


Translation Examples:
Input: 'i am happy'
Output: je suis un étudiant
Input: 'you are smart'
Output: tu es intelligent


## Transformer

### Transformer - MT

DAta:

source	target

Hello, how are you?	Bonjour, comment ça va ?

What is your name?	Quel est ton nom ?

I am learning machine translation.	J'apprends la traduction automatique.

I love programming.	J'aime la programmation.

How old are you?	Quel âge as-tu ?

French	Helsinki-NLP/opus-mt-en-fr

German	Helsinki-NLP/opus-mt-en-de

Spanish	Helsinki-NLP/opus-mt-en-es

Italian	Helsinki-NLP/opus-mt-en-it

Portuguese	Helsinki-NLP/opus-mt-en-pt

Russian	Helsinki-NLP/opus-mt-en-ru

Chinese (Simplified)	Helsinki-NLP/opus-mt-en-zh

Japanese	Helsinki-NLP/opus-mt-en-jap

Hindi	Helsinki-NLP/opus-mt-en-hi

Arabic	Helsinki-NLP/opus-mt-en-ar

Bengali	Helsinki-NLP/opus-mt-en-bn

Tamil	Helsinki-NLP/opus-mt-en-ta

Urdu	Helsinki-NLP/opus-mt-en-ur

In [68]:
import pandas as pd
from datasets import Dataset
from transformers import (
    MarianTokenizer,
    MarianMTModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

# Config
model_name = "Helsinki-NLP/opus-mt-en-fr"
batch_size = 4
epochs = 3

# Load dataset (CSV with 'src' and 'tgt' columns)
df = pd.read_csv("translation.txt",sep="\t")  # Your dataset here
print(df)
dataset = Dataset.from_pandas(df)

# Load model and tokenizer
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Preprocessing
def preprocess(example):
    model_inputs = tokenizer(example["source"], max_length=128, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(example["target"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./finetuned-en-fr",
    evaluation_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    num_train_epochs=epochs,
    weight_decay=0.01,
    save_total_limit=1,
    logging_dir='./logs',
    report_to="none"  # Disable WandB, Comet, etc.
)

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Fine-tune the model
trainer.train()

# Save model
model.save_pretrained("finetuned-marian-en-fr")
tokenizer.save_pretrained("finetuned-marian-en-fr")

                               source                                 target
0                 Hello, how are you?               Bonjour, comment ça va ?
1                  What is your name?                     Quel est ton nom ?
2  I am learning machine translation.  J'apprends la traduction automatique.
3                 I love programming.               J'aime la programmation.
4                    How old are you?                       Quel âge as-tu ?


/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-68-3fc8facc4709>:52: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3353: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[59513]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('finetuned-marian-en-fr/tokenizer_config.json',
 'finetuned-marian-en-fr/special_tokens_map.json',
 'finetuned-marian-en-fr/vocab.json',
 'finetuned-marian-en-fr/source.spm',
 'finetuned-marian-en-fr/target.spm',
 'finetuned-marian-en-fr/added_tokens.json')

In [69]:
from transformers import MarianMTModel, MarianTokenizer

def translate(text, model_dir="finetuned-marian-en-fr"):
    tokenizer = MarianTokenizer.from_pretrained(model_dir)
    model = MarianMTModel.from_pretrained(model_dir)
    encoded = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    generated = model.generate(**encoded)
    return tokenizer.decode(generated[0], skip_special_tokens=True)

# Example
print(translate("Hi, I am deeksha"))

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Bonjour, je suis deeksha.


In [73]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.2 MB/s eta 0:00:00


In [74]:
import nltk
nltk.download('punkt')
import evaluate

# Load BLEU metric
bleu = evaluate.load("bleu")

# Example list of predictions and references
predictions = ["Je t'aime beaucoup."]
references = [["Je vous aime beaucoup."]]  # note: reference must be a list of lists

# Compute BLEU
results = bleu.compute(predictions=predictions, references=references)
print(f"BLEU score: {results['bleu']:.4f}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


BLEU score: 0.0000


### Transformer-Text summarization

In [42]:
import pandas as pd
from datasets import Dataset
from transformers import BartForConditionalGeneration, BartTokenizer, TrainingArguments, Trainer

# Load a smaller dataset sample
data = pd.read_csv("summary.txt",sep = '\t')
sample_size = min(100, len(data))
data = data.sample(n=sample_size, random_state=42)
dataset = Dataset.from_pandas(data)

# Use smaller DistilBART model
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# Preprocess
def preprocess_function(examples):
    inputs = tokenizer(examples["text"], max_length=512, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["summary"], max_length=128, truncation=True, padding="max_length")
    inputs["labels"] = labels["input_ids"]
    return inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Training arguments (CPU optimized)
training_args = TrainingArguments(
    output_dir="./finetuned_bart_summary_cpu",
    evaluation_strategy="no",  # skip eval
    learning_rate=5e-5,
    per_device_train_batch_size=2,  # keep low for CPU
    num_train_epochs=3,  # 1 epoch for testing
    logging_strategy="no",
    save_strategy="no",
    report_to="none"  # no wandb or hub
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer
)

# Train (fast on CPU)
trainer.train()

# Save the model
model.save_pretrained("finetuned-bart-summary-cpu")
tokenizer.save_pretrained("finetuned-bart-summary-cpu")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-42-e3a6662312b9>:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3353: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('finetuned-bart-summary-cpu/tokenizer_config.json',
 'finetuned-bart-summary-cpu/special_tokens_map.json',
 'finetuned-bart-summary-cpu/vocab.json',
 'finetuned-bart-summary-cpu/merges.txt',
 'finetuned-bart-summary-cpu/added_tokens.json')

In [43]:
from transformers import pipeline

def summarize(text, model_path="finetuned-bart-summary-cpu"):
    summarization_model = pipeline(
        "summarization",
        model=model_path,
        tokenizer=model_path
    )

    # Generate the summary
    summary = summarization_model(text)[0]['summary_text']
    return summary

# Example
text = '''Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit instructions.
          Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
          ML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine.The application of ML to business problems is known as predictive analytics.
          Statistics and mathematical optimization (mathematical programming) methods comprise the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) via unsupervised learning.
          From a theoretical viewpoint, probably approximately correct learning provides a framework for describing machine learning.'''

print(summarize(text))

/usr/local/lib/python3.11/dist-packages/transformers/models/bart/configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(
Device set to use cpu


 Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data . ML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine .


### Transformer-Sentiment

Sentiment.csv

text,label

"I love this product",positive

"This is terrible",negative

"Absolutely fantastic experience",positive

"I would never buy this again",negative

"Great quality and fast shipping",positive

"The item broke after one use",negative

"Very satisfied with my purchase",positive

"Totally disappointed with the service",negative

"Exceeded my expectations!",positive

"Not worth the money at all",negative


In [6]:
!pip install transformers datasets scikit-learn

In [47]:
import pandas as pd
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import torch

# Load and prepare dataset
df = pd.read_csv("sentiment.txt")  # Replace with your file path
label_map = {"negative": 0, "positive": 1}
df["label"] = df["label"].map(label_map)

dataset = Dataset.from_pandas(df)

# Split into train and test
dataset = dataset.train_test_split(test_size=0.1)

# Load tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Preprocessing
def preprocess(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

tokenized = dataset.map(preprocess, batched=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="./finetuned-sentiment-model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="epoch",
    report_to="none"  # disables wandb
)

# Data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train
trainer.train()

# Save model and tokenizer
model.save_pretrained("finetuned-sentiment-model")
tokenizer.save_pretrained("finetuned-sentiment-model")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-47-5c71a7fa6a36>:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,No log,0.675201
2,No log,0.689477
3,No log,0.696787


('finetuned-sentiment-model/tokenizer_config.json',
 'finetuned-sentiment-model/special_tokens_map.json',
 'finetuned-sentiment-model/vocab.txt',
 'finetuned-sentiment-model/added_tokens.json',
 'finetuned-sentiment-model/tokenizer.json')

In [48]:
from transformers import TextClassificationPipeline

pipe = TextClassificationPipeline(model=model, tokenizer=tokenizer, top_k=None)

print(pipe("I love this!"))
print(pipe("This is bad."))

Device set to use cpu


[[{'label': 'LABEL_1', 'score': 0.5355250239372253}, {'label': 'LABEL_0', 'score': 0.46447497606277466}]]
[[{'label': 'LABEL_1', 'score': 0.5142375826835632}, {'label': 'LABEL_0', 'score': 0.485762357711792}]]
